# ADD 2026 Track 2 - Qwen2.5-Omni Baseline Dev Evaluation

This notebook downloads the baseline code archive, released dataset, and official checkpoint from Google Drive, runs a smoke test, performs restart-safe sharded inference on all 2,000 Dev samples, and evaluates both Official Dev and Clean Dev.

Recommended runtime: NVIDIA A100. Start with `MAX_BATCH_SIZE=1`; increase only after the smoke test succeeds. T4 is not supported by this notebook because the baseline uses BF16 and a 7B multimodal model.

All persistent outputs are written to Google Drive. The downloaded dataset must not be redistributed.

## 1. Configuration
Edit only this cell when paths or inference settings change.

In [ ]:
from pathlib import Path
import os, json, time, shutil, hashlib, zipfile, tarfile, re

PROJECT_DRIVE_FILE_ID = "1WhUdcUB6dDW6fKtcMqDaF-GNzGwi1abJ"
DATASET_DRIVE_FILE_ID = "1QQIEf5lE5I5dIpKrnxGW8noj71zxZmd8"
CHECKPOINT_DRIVE_FILE_ID = "1_Tr_Bpm5mmTyIQi2Px7ATyXDPENqXIxp"

CONTENT_ROOT = Path("/content")
DOWNLOAD_ROOT = CONTENT_ROOT / "add2026_downloads"
EXTRACT_ROOT = CONTENT_ROOT / "add2026_project_payload"
VENV_DIR = CONTENT_ROOT / "add2026_py311"
SCRATCH_ROOT = CONTENT_ROOT / "add2026_eval_scratch"
DRIVE_RESULTS_DIR = Path("/content/drive/MyDrive/ADD2026/results/qwen25_baseline_dev")

PROJECT_DOWNLOAD = DOWNLOAD_ROOT / "project_archive"
DATASET_DOWNLOAD = DOWNLOAD_ROOT / "dataset_archive"
CHECKPOINT_DOWNLOAD = DOWNLOAD_ROOT / "track2_baseline_weights.zip"
SHARD_SIZE = 500
SMOKE_SAMPLES = 8
MAX_BATCH_SIZE = 1
MAX_NEW_TOKENS = 2048
CLEAN_DEV_EXCLUDED_IDS = {"ADD2026_T2_EAD_dev_000621"}

for path in (DOWNLOAD_ROOT, SCRATCH_ROOT):
    path.mkdir(parents=True, exist_ok=True)

print({
    "project_file_id": PROJECT_DRIVE_FILE_ID,
    "dataset_file_id": DATASET_DRIVE_FILE_ID,
    "checkpoint_file_id": CHECKPOINT_DRIVE_FILE_ID,
    "results_dir": str(DRIVE_RESULTS_DIR),
    "shard_size": SHARD_SIZE,
    "max_batch_size": MAX_BATCH_SIZE,
})

## 2. Mount Drive and inspect the GPU
The notebook stops early if no NVIDIA GPU is attached.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE_RESULTS_DIR.mkdir(parents=True, exist_ok=True)
!nvidia-smi
gpu_check = !nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
if not gpu_check:
    raise RuntimeError("No NVIDIA GPU detected. Select a GPU runtime and reconnect.")
print("GPU:", gpu_check[0])

## 3. Install download tools and create an isolated Python 3.11 environment
The baseline pins PyTorch 2.6.0 with CUDA 12.4. The notebook uses a separate Python 3.11 environment even if Colab's default Python changes. Installation normally takes 10-20 minutes.

In [ ]:
!python -m pip install -q gdown uv
!uv python install 3.11
!uv venv --python 3.11 {VENV_DIR}
PYTHON_BIN = VENV_DIR / "bin/python"
SWIFT_BIN = VENV_DIR / "bin/swift"
print("Python binary:", PYTHON_BIN)
!{PYTHON_BIN} --version

## 4. Download the code and dataset archives
The Drive file `1WhU...` is a small `track2_baseline.zip` containing the baseline code. The released `train/`, `dev/`, and `labels/` data are downloaded separately from `1QQI...`. The dataset archive is extracted to Colab's local SSD; only results are written back to your Drive.

In [ ]:
project_url = f"https://drive.google.com/file/d/{PROJECT_DRIVE_FILE_ID}/view"
if not PROJECT_DOWNLOAD.exists():
    !gdown --fuzzy {project_url} -O {PROJECT_DOWNLOAD}
else:
    print("Reusing downloaded project archive:", PROJECT_DOWNLOAD)

if not PROJECT_DOWNLOAD.is_file() or PROJECT_DOWNLOAD.stat().st_size == 0:
    raise RuntimeError("Project archive download failed or is empty.")

if not EXTRACT_ROOT.exists():
    EXTRACT_ROOT.mkdir(parents=True)
    if zipfile.is_zipfile(PROJECT_DOWNLOAD):
        with zipfile.ZipFile(PROJECT_DOWNLOAD) as zf:
            zf.extractall(EXTRACT_ROOT)
    elif tarfile.is_tarfile(PROJECT_DOWNLOAD):
        with tarfile.open(PROJECT_DOWNLOAD) as tf:
            tf.extractall(EXTRACT_ROOT, filter="data")
    else:
        raise RuntimeError("The downloaded project file is not a supported ZIP/TAR archive.")

project_candidates = sorted(p.parent for p in EXTRACT_ROOT.rglob("prepare_data.py"))
if len(project_candidates) != 1:
    raise RuntimeError(f"Expected one project root, found: {project_candidates}")
PROJECT_DIR = project_candidates[0]
print("Project root:", PROJECT_DIR)
print("Project archive size (MiB):", round(PROJECT_DOWNLOAD.stat().st_size / 2**20, 3))

def progress(done, total, start, label):
    elapsed = max(time.time() - start, 1e-6)
    eta = elapsed * (total - done) / done if done else 0
    print(f"[{label}] {done}/{total} ({100 * done / total:.1f}%) elapsed={elapsed:.1f}s ETA={eta:.1f}s", flush=True)

dataset_url = f"https://drive.google.com/file/d/{DATASET_DRIVE_FILE_ID}/view"
if not DATASET_DOWNLOAD.exists():
    !gdown --fuzzy {dataset_url} -O {DATASET_DOWNLOAD}
else:
    print("Reusing downloaded dataset archive:", DATASET_DOWNLOAD)
if not DATASET_DOWNLOAD.is_file() or DATASET_DOWNLOAD.stat().st_size == 0:
    raise RuntimeError("Dataset archive download failed or is empty.")

DATA_PAYLOAD_ROOT = CONTENT_ROOT / "add2026_data_payload"
required_parts = ("train", "dev", "labels")
existing_data = PROJECT_DIR / "data"
if not all((existing_data / part).is_dir() for part in required_parts):
    if not DATA_PAYLOAD_ROOT.exists():
        DATA_PAYLOAD_ROOT.mkdir(parents=True)
        start = time.time()
        if zipfile.is_zipfile(DATASET_DOWNLOAD):
            with zipfile.ZipFile(DATASET_DOWNLOAD) as zf:
                members = zf.infolist()
                for index, member in enumerate(members, 1):
                    zf.extract(member, DATA_PAYLOAD_ROOT)
                    if index % 500 == 0 or index == len(members):
                        progress(index, len(members), start, "extract-data")
        else:
            raise RuntimeError("The downloaded dataset is not a valid ZIP archive. Check Drive access/quota.")

    source_data = DATA_PAYLOAD_ROOT / "track2_training_datasets"
    if not all((source_data / part).is_dir() for part in required_parts):
        raise RuntimeError(f"Expected train/dev/labels under {source_data}")
    existing_data.mkdir(parents=True, exist_ok=True)
    for part in required_parts:
        target = existing_data / part
        if not target.exists():
            target.symlink_to(source_data / part, target_is_directory=True)
    print("Attached local dataset root:", source_data)

## 5. Validate the released dataset before installing the model stack

In [ ]:
def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8-sig").splitlines() if line.strip()]

expected = {"train": 8600, "dev": 2000}
for split, count in expected.items():
    wavs = sorted((PROJECT_DIR / "data" / split).glob("*.wav"))
    labels = read_jsonl(PROJECT_DIR / "data" / "labels" / f"{split}_labels.jsonl")
    wav_ids = {p.stem for p in wavs}
    label_ids = {r["file_id"] for r in labels}
    assert len(wavs) == count, (split, len(wavs), count)
    assert len(labels) == count, (split, len(labels), count)
    assert wav_ids == label_ids, f"{split}: WAV/label ID mismatch"
    counts = {label: sum(r["label"] == label for r in labels) for label in ("real", "fake")}
    print(split, {"wavs": len(wavs), "labels": len(labels), **counts})

## 6. Install the pinned baseline dependencies
This uses the repository's exact `requirements.txt` and ms-swift revision. The output is intentionally verbose so installation failures remain diagnosable.

In [ ]:
!uv pip install --python {PYTHON_BIN} torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
!uv pip install --python {PYTHON_BIN} -r {PROJECT_DIR / 'requirements.txt'}
os.environ["PATH"] = f"{VENV_DIR / 'bin'}:{os.environ['PATH']}"
os.environ["PYTHONUNBUFFERED"] = "1"
!{PYTHON_BIN} -c "import torch; print('torch', torch.__version__, 'cuda', torch.version.cuda, 'available', torch.cuda.is_available())"
!{SWIFT_BIN} --help | head -n 20

## 7. Download, verify, and extract the official baseline checkpoint

In [ ]:
checkpoint_url = f"https://drive.google.com/file/d/{CHECKPOINT_DRIVE_FILE_ID}/view"
if not CHECKPOINT_DOWNLOAD.exists():
    !gdown --fuzzy {checkpoint_url} -O {CHECKPOINT_DOWNLOAD}
else:
    print("Reusing downloaded checkpoint archive:", CHECKPOINT_DOWNLOAD)

if not zipfile.is_zipfile(CHECKPOINT_DOWNLOAD):
    raise RuntimeError("Checkpoint download is not a valid ZIP archive. Check Drive access/quota.")

checkpoint_hash = hashlib.sha256()
with CHECKPOINT_DOWNLOAD.open("rb") as stream:
    for chunk in iter(lambda: stream.read(1024 * 1024), b""):
        checkpoint_hash.update(chunk)
checkpoint_sha256 = checkpoint_hash.hexdigest()
print("Checkpoint SHA-256:", checkpoint_sha256)

CHECKPOINT_EXTRACT_ROOT = CONTENT_ROOT / "add2026_checkpoint"
if not CHECKPOINT_EXTRACT_ROOT.exists():
    CHECKPOINT_EXTRACT_ROOT.mkdir(parents=True)
    with zipfile.ZipFile(CHECKPOINT_DOWNLOAD) as zf:
        zf.extractall(CHECKPOINT_EXTRACT_ROOT)

config_candidates = sorted(CHECKPOINT_EXTRACT_ROOT.rglob("config.json"))
weight_roots = [p.parent for p in config_candidates if list(p.parent.glob("*.safetensors")) or list(p.parent.glob("*.bin"))]
if len(weight_roots) != 1:
    raise RuntimeError(f"Expected one checkpoint directory, found: {weight_roots}")
CHECKPOINT_DIR = weight_roots[0]
print("Checkpoint directory:", CHECKPOINT_DIR)

## 8. Build ms-swift manifests and run a smoke test
The smoke test uses eight Dev files. Do not start the full run unless it produces eight valid XML responses without OOM.

In [ ]:
%cd {PROJECT_DIR}
!{PYTHON_BIN} prepare_data.py
swift_dev = read_jsonl(PROJECT_DIR / "data/swift/dev.jsonl")
assert len(swift_dev) == 2000
print("Prepared Dev manifest:", len(swift_dev))

In [ ]:
SMOKE_AUDIO_DIR = SCRATCH_ROOT / "smoke_wavs"
SMOKE_OUTPUT_DIR = SCRATCH_ROOT / "smoke_output"
SMOKE_AUDIO_DIR.mkdir(parents=True, exist_ok=True)
for wav in sorted((PROJECT_DIR / "data/dev").glob("*.wav"))[:SMOKE_SAMPLES]:
    link = SMOKE_AUDIO_DIR / wav.name
    if not link.exists():
        link.symlink_to(wav)

raw_smoke = SMOKE_OUTPUT_DIR / "raw_predictions.jsonl"
if raw_smoke.exists() and sum(1 for line in raw_smoke.open() if line.strip()) == SMOKE_SAMPLES:
    print("Smoke test already complete; skipping inference.")
else:
    if SMOKE_OUTPUT_DIR.exists():
        archived = SMOKE_OUTPUT_DIR.with_name(f"smoke_output.partial_{int(time.time())}")
        SMOKE_OUTPUT_DIR.rename(archived)
        print("Archived incomplete smoke output to", archived)
    os.environ["CHECKPOINT_DIR"] = str(CHECKPOINT_DIR)
    os.environ["SMOKE_AUDIO_DIR"] = str(SMOKE_AUDIO_DIR)
    os.environ["SMOKE_OUTPUT_DIR"] = str(SMOKE_OUTPUT_DIR)
    os.environ["MAX_BATCH_SIZE"] = str(MAX_BATCH_SIZE)
    os.environ["MAX_NEW_TOKENS"] = str(MAX_NEW_TOKENS)
    !bash infer.sh "$CHECKPOINT_DIR" "$SMOKE_AUDIO_DIR" "$SMOKE_OUTPUT_DIR"

smoke_rows = read_jsonl(raw_smoke)
assert len(smoke_rows) == SMOKE_SAMPLES
valid_xml = sum(
    len(re.findall(r"<Decision>\s*(?:real|fake)\s*</Decision>", row.get("response", ""), re.I)) == 1
    and len(re.findall(r"<Evidence>.*?</Evidence>", row.get("response", ""), re.I | re.S)) == 1
    for row in smoke_rows
)
print({"smoke_rows": len(smoke_rows), "valid_xml": valid_xml})
if valid_xml != SMOKE_SAMPLES:
    raise RuntimeError("Smoke test completed but not every response contains exactly one Decision and Evidence field.")

## 9. Create restart-safe Dev shards
Four 500-file shards are created with symbolic links. Completed shard outputs in Drive are skipped on rerun.

In [ ]:
SHARDS_ROOT = SCRATCH_ROOT / "dev_shards"
SHARDS_ROOT.mkdir(parents=True, exist_ok=True)
dev_wavs = sorted((PROJECT_DIR / "data/dev").glob("*.wav"))
num_shards = (len(dev_wavs) + SHARD_SIZE - 1) // SHARD_SIZE
for shard_index in range(num_shards):
    shard_dir = SHARDS_ROOT / f"shard_{shard_index:03d}"
    shard_dir.mkdir(exist_ok=True)
    members = dev_wavs[shard_index * SHARD_SIZE:(shard_index + 1) * SHARD_SIZE]
    for wav in members:
        link = shard_dir / wav.name
        if not link.exists():
            link.symlink_to(wav)
    assert len(list(shard_dir.glob("*.wav"))) == len(members)
    print(shard_dir.name, len(members))

os.environ.update({
    "ADD_PROJECT_DIR": str(PROJECT_DIR),
    "ADD_CHECKPOINT_DIR": str(CHECKPOINT_DIR),
    "ADD_SHARDS_ROOT": str(SHARDS_ROOT),
    "ADD_RESULTS_DIR": str(DRIVE_RESULTS_DIR),
    "ADD_NUM_SHARDS": str(num_shards),
    "ADD_MAX_BATCH_SIZE": str(MAX_BATCH_SIZE),
    "ADD_MAX_NEW_TOKENS": str(MAX_NEW_TOKENS),
})

## 10. Run full Dev inference
This is the long-running cell. ms-swift displays batch progress; the wrapper additionally reports shard completion, throughput, and updated ETA. Incomplete outputs are preserved with a `.partial_TIMESTAMP` suffix before retrying.

In [ ]:
%%bash
set -euo pipefail
export PATH="/content/add2026_py311/bin:$PATH"
export PYTHONUNBUFFERED=1
export MAX_BATCH_SIZE="$ADD_MAX_BATCH_SIZE"
export MAX_NEW_TOKENS="$ADD_MAX_NEW_TOKENS"
mkdir -p "$ADD_RESULTS_DIR"
gpu_log="$ADD_RESULTS_DIR/gpu_usage.csv"
nvidia-smi --query-gpu=timestamp,name,memory.used,memory.total,utilization.gpu --format=csv -l 5 > "$gpu_log" &
monitor_pid=$!
trap 'kill $monitor_pid 2>/dev/null || true' EXIT

overall_start=$(date +%s)
completed=0
for ((i=0; i<ADD_NUM_SHARDS; i++)); do
  shard=$(printf 'shard_%03d' "$i")
  audio_dir="$ADD_SHARDS_ROOT/$shard"
  output_dir="$ADD_RESULTS_DIR/$shard"
  expected=$(find "$audio_dir" -maxdepth 1 -name '*.wav' | wc -l | tr -d ' ')
  actual=0
  if [[ -f "$output_dir/raw_predictions.jsonl" ]]; then
    actual=$(grep -cve '^$' "$output_dir/raw_predictions.jsonl" || true)
  fi
  if [[ "$actual" == "$expected" ]]; then
    echo "[skip] $shard already complete ($actual/$expected)"
  else
    if [[ -d "$output_dir" ]]; then
      mv "$output_dir" "${output_dir}.partial_$(date +%s)"
    fi
    shard_start=$(date +%s)
    echo "[start] $shard ($expected samples) at $(date -Is)"
    cd "$ADD_PROJECT_DIR"
    bash infer.sh "$ADD_CHECKPOINT_DIR" "$audio_dir" "$output_dir"
    shard_end=$(date +%s)
    actual=$(grep -cve '^$' "$output_dir/raw_predictions.jsonl" || true)
    [[ "$actual" == "$expected" ]] || { echo "[error] $shard produced $actual/$expected rows"; exit 1; }
    elapsed=$((shard_end - shard_start))
    echo "[done] $shard elapsed=${elapsed}s rate=$(awk -v n="$actual" -v t="$elapsed" 'BEGIN {printf "%.4f", n/t}') samples/s"
  fi
  completed=$((completed + 1))
  now=$(date +%s)
  elapsed_total=$((now - overall_start))
  remaining=$((ADD_NUM_SHARDS - completed))
  eta=$((elapsed_total * remaining / completed))
  echo "[progress] shards=$completed/$ADD_NUM_SHARDS elapsed=${elapsed_total}s ETA=${eta}s"
done


## 11. Merge shards and normalize audio keys
Shard paths are rewritten to canonical `data/dev/<file>.wav` paths so `eval_add2026.py` can match all predictions to the official Dev manifest.

In [ ]:
merged_inputs = []
merged_raw = []
for shard_index in range(num_shards):
    shard_output = DRIVE_RESULTS_DIR / f"shard_{shard_index:03d}"
    inputs = read_jsonl(shard_output / "input.jsonl")
    raw = read_jsonl(shard_output / "raw_predictions.jsonl")
    assert len(inputs) == len(raw)
    for row in inputs:
        fid = row.get("id") or Path(row["audios"][0]).stem
        row["id"] = fid
        row["audios"] = [f"data/dev/{fid}.wav"]
    for row in raw:
        fid = Path(row["audios"][0]).stem
        row["audios"] = [f"data/dev/{fid}.wav"]
    merged_inputs.extend(inputs)
    merged_raw.extend(raw)

def write_jsonl(path, rows):
    path = Path(path)
    with path.open("w", encoding="utf-8", newline="\n") as stream:
        for row in rows:
            stream.write(json.dumps(row, ensure_ascii=False) + "\n")

input_ids = [r["id"] for r in merged_inputs]
raw_ids = [Path(r["audios"][0]).stem for r in merged_raw]
assert len(merged_inputs) == len(merged_raw) == 2000
assert len(set(input_ids)) == len(set(raw_ids)) == 2000
assert set(input_ids) == set(raw_ids)

MERGED_INPUT = DRIVE_RESULTS_DIR / "input.jsonl"
MERGED_RAW = DRIVE_RESULTS_DIR / "raw_predictions.jsonl"
PREDICTION_JSONL = DRIVE_RESULTS_DIR / "prediction.jsonl"
write_jsonl(MERGED_INPUT, merged_inputs)
write_jsonl(MERGED_RAW, merged_raw)
print({"merged_inputs": len(merged_inputs), "merged_predictions": len(merged_raw)})

os.environ["MERGED_INPUT"] = str(MERGED_INPUT)
os.environ["MERGED_RAW"] = str(MERGED_RAW)
os.environ["PREDICTION_JSONL"] = str(PREDICTION_JSONL)
!{PYTHON_BIN} export_submission.py --input "$MERGED_INPUT" --raw "$MERGED_RAW" --output "$PREDICTION_JSONL"

## 12. Evaluate Official Dev and Clean Dev
Classification-only metrics run first. BERTScore then runs on GPU after the inference process has released model memory.

In [ ]:
OFFICIAL_SUBSET = PROJECT_DIR / "data/swift/dev.jsonl"
official_rows = read_jsonl(OFFICIAL_SUBSET)
clean_rows = [row for row in official_rows if row["id"] not in CLEAN_DEV_EXCLUDED_IDS]
CLEAN_SUBSET = DRIVE_RESULTS_DIR / "dev_clean.jsonl"
write_jsonl(CLEAN_SUBSET, clean_rows)
assert len(official_rows) == 2000
assert len(clean_rows) == 1999

OFFICIAL_ACC = DRIVE_RESULTS_DIR / "official_dev_accuracy.json"
OFFICIAL_EVAL = DRIVE_RESULTS_DIR / "official_dev_eval.json"
CLEAN_EVAL = DRIVE_RESULTS_DIR / "clean_dev_eval.json"
for name, value in {
    "OFFICIAL_SUBSET": OFFICIAL_SUBSET,
    "CLEAN_SUBSET": CLEAN_SUBSET,
    "OFFICIAL_ACC": OFFICIAL_ACC,
    "OFFICIAL_EVAL": OFFICIAL_EVAL,
    "CLEAN_EVAL": CLEAN_EVAL,
}.items():
    os.environ[name] = str(value)

!{PYTHON_BIN} eval_add2026.py "$MERGED_RAW" --subset "$OFFICIAL_SUBSET" --no-bert --out-json "$OFFICIAL_ACC"
!{PYTHON_BIN} eval_add2026.py "$MERGED_RAW" --subset "$OFFICIAL_SUBSET" --bert-device cuda:0 --out-json "$OFFICIAL_EVAL"
!{PYTHON_BIN} eval_add2026.py "$MERGED_RAW" --subset "$CLEAN_SUBSET" --bert-device cuda:0 --out-json "$CLEAN_EVAL"

## 13. Final validation and experiment summary

In [ ]:
official = json.loads(OFFICIAL_EVAL.read_text(encoding="utf-8"))
clean = json.loads(CLEAN_EVAL.read_text(encoding="utf-8"))
responses = [row.get("response", "") for row in merged_raw]
xml_valid = [
    len(re.findall(r"<Decision>\s*(?:real|fake)\s*</Decision>", response, re.I)) == 1
    and len(re.findall(r"<Evidence>.*?</Evidence>", response, re.I | re.S)) == 1
    for response in responses
]
evidence_words = []
for response in responses:
    match = re.search(r"<Evidence>(.*?)</Evidence>", response, re.I | re.S)
    text = re.sub(r"<[^>]+>", " ", match.group(1)) if match else ""
    evidence_words.append(len(re.sub(r"\s+", " ", text).strip().split()))

summary = {
    "checkpoint_drive_file_id": CHECKPOINT_DRIVE_FILE_ID,
    "checkpoint_sha256": checkpoint_sha256,
    "project_drive_file_id": PROJECT_DRIVE_FILE_ID,
    "dataset_drive_file_id": DATASET_DRIVE_FILE_ID,
    "official_dev": {
        "samples": official["subset_total"],
        "real_recall": official["real"]["recall"],
        "fake_recall": official["fake"]["recall"],
        "balanced_accuracy": official["balanced_accuracy"],
        "overall_accuracy": official["overall_accuracy"],
        "real_joint": official["real"]["joint"],
        "fake_joint": official["fake"]["joint"],
        "final_score_100": official["final_score_100"],
    },
    "clean_dev": {
        "samples": clean["subset_total"],
        "balanced_accuracy": clean["balanced_accuracy"],
        "overall_accuracy": clean["overall_accuracy"],
        "final_score_100": clean["final_score_100"],
    },
    "xml_validity_rate": sum(xml_valid) / len(xml_valid),
    "mean_evidence_words": sum(evidence_words) / len(evidence_words),
    "max_batch_size": MAX_BATCH_SIZE,
    "max_new_tokens": MAX_NEW_TOKENS,
}
SUMMARY_PATH = DRIVE_RESULTS_DIR / "summary.json"
SUMMARY_PATH.write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
print(json.dumps(summary, ensure_ascii=False, indent=2))
print("Saved all persistent artifacts to:", DRIVE_RESULTS_DIR)